# Raleigh dataset

In [ ]:
import sys
import subprocess

sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import pandas as pd
import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

In [ ]:
epsg = "6542"
gs.create_project(f"raleigh_northcarolina_usa_epsg{epsg}", epsg=epsg)

In [ ]:
gj.init(f"raleigh_northcarolina_usa_epsg{epsg}")

In [ ]:
tools = Tools()

In [ ]:
tools.g_region(n=228500, s=215000, w=629000, e=653000, res=10, flags="s")

## Utils

In [ ]:
def bbox_latlon():
    """Current computational region as a WGS84 (west, south, east, north) tuple."""
    b = tools.g_region(flags="b", format="json")
    return (b["ll_w"], b["ll_s"], b["ll_e"], b["ll_n"])

## Elevation

USGS 3DEP elevation (1/3 arc second resolution ~ 10 m )

In [ ]:
elevation = "elevation"

In [ ]:
tools.g_extension(extension="r.in.usgs")

In [ ]:
tools.r_in_usgs(product="ned", output_name=elevation, ned_dataset="ned13sec", flags="i", ned_release="all", verbose=True)

In [ ]:
tools.r_in_usgs(product="ned", output_name=elevation, ned_dataset="ned13sec", ned_release="all", title_filter="20250507")

In [ ]:
elev_map = gj.Map(width=800)
elev_map.d_rast(map=elevation)
elev_map.show()

In [ ]:
tools.r_support(map=elevation, title="USGS 3DEP elevation (1/3 arc second)", source1="USGS")

In [ ]:
tools.r_info(map=elevation)

## Landuse

In [ ]:
landuse = "landuse"

In [ ]:
import os
import urllib.request
import zipfile
from pathlib import Path

url = "https://www.mrlc.gov/downloads/sciweb1/shared/mrlc/data-bundles/Annual_NLCD_LndCov_2024_CU_C1V1.zip"
nlcd_filename, headers = urllib.request.urlretrieve(url)
with zipfile.ZipFile(nlcd_filename, "r") as zip_ref:
    zip_ref.extractall()
os.remove(nlcd_filename)
nlcd_filename = Path(url).with_suffix(".tif").name

In [ ]:
# create a project
gs.create_project("nlcd", filename=nlcd_filename)
# initialize GRASS session in that project
with gs.setup.init("nlcd", env=os.environ.copy()) as session:
    # Run GRASS tools
    Tools(session=session).r_external(input=nlcd_filename, output="nlcd")

In [ ]:
tools.r_proj(project="nlcd", mapset="PERMANENT", input="nlcd", output=landuse, resolution=30)

In [ ]:
import io

categories = \
"""
0:Unclassified
11:Open Water
12:Perennial Snow/Ice
21:Developed, Open Space
22:Developed, Low Intensity
23:Developed, Medium Intensity
24:Developed, High Intensity
31:Barren Land
41:Deciduous Forest
42:Evergreen Forest
43:Mixed Forest
52:Shrub/Scrub
71:Grasslands/Herbaceous
81:Pasture/Hay
82:Cultivated Crops
90:Woody Wetlands
95:Emergent Herbaceous Wetlands
"""
tools.r_category(map=landuse, rules=io.StringIO(categories), separator=":")

In [ ]:
tools.r_support(map=landuse, title="USGS National Land Cover Data 2024", source1="USGS")

In [ ]:
tools.r_colors(map=landuse, color="nlcd")
nlcd_map = gj.Map(width=800)
nlcd_map.d_rast(map=landuse)
nlcd_map.d_legend(raster=landuse, flags="ncb")
nlcd_map.show()

In [ ]:
tools.r_info(map=landuse)

## Landsat

In [ ]:
!pip install pystac_client planetary_computer

In [ ]:
from pystac_client import Client

catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox_latlon(),
    datetime="2025-03-01/2025-07-01",
    query={"eo:cloud_cover": {"lt": 10},
           "platform": {"eq": "landsat-8"}},
)
items = search.item_collection()
selected_item = min(items, key=lambda item: item.properties["eo:cloud_cover"])
selected_item

In [ ]:
from IPython.display import Image
Image(url=selected_item.assets["rendered_preview"].href, width=600)

In [ ]:
import requests
import planetary_computer

keys = ("coastal", "blue", "green", "red", "nir08", "swir16", "swir22")
for k in keys:
    url = planetary_computer.sign(selected_item).assets[k].href
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(f"{k}.tif", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

In [ ]:
from datetime import datetime

def grass_timestamp(iso_string):
    """Convert an ISO timestamp to the format r.timestamp expects."""
    dt = datetime.fromisoformat(iso_string.replace("Z", "+00:00"))
    return dt.strftime("%d %b %Y %H:%M:%S %z").strip()

timestamp = grass_timestamp(selected_item.properties["datetime"])

for band, k in enumerate(keys, start=1):
    name = f"landsat8_2025_B{band}"
    tools.r_import(input=f"{k}.tif", output=name, extent="region",
                   resolution="value", resolution_value=30)
    tools.r_semantic_label(map=name, semantic_label=f"L8_{band}")
    tools.r_support(map=name, title=selected_item.assets[k].description)
    tools.r_timestamp(map=name, date=timestamp)

In [ ]:
tools.i_colors_enhance(blue="landsat8_2025_B2", green="landsat8_2025_B3", red="landsat8_2025_B4", strength=98, flags="p")
landsat_map = gj.Map(width=800)
landsat_map.d_rgb(red="landsat8_2025_B4", green="landsat8_2025_B3", blue="landsat8_2025_B2")
landsat_map.show()

## Roads

In [ ]:
roads = "roads"

In [ ]:
!pip install osmnx

In [ ]:
import osmnx as ox

G = ox.graph_from_bbox(bbox=bbox_latlon(), network_type="drive")
nodes, edges = ox.graph_to_gdfs(G)

# OSM allows several road classes per way; keep the most important one
rank = {
    "motorway": 0, "trunk": 1, "primary": 2, "secondary": 3,
    "tertiary": 4, "unclassified": 5, "residential": 6,
    "living_street": 7, "crossing": 8,
    "motorway_link": 0, "trunk_link": 1, "primary_link": 2,
    "secondary_link": 3, "tertiary_link": 4,
}
edges["highway"] = edges["highway"].apply(
    lambda v: min(v, key=lambda t: rank.get(t, 99)) if isinstance(v, list) else v
)

edges.to_file("roads.gpkg")

In [ ]:
tools.v_import(input="roads.gpkg", output=roads, snap=1e-8)

In [ ]:
tools.v_db_dropcolumn(map=roads, columns=["u", "v", "key"])

In [ ]:
tools.v_support(map="roads", map_name="OSM roads",
                comment="OSM roads downloaded using the OSMnx Python package",
                map_date=grass_timestamp(datetime.today().isoformat()))

In [ ]:
road_map = gj.Map(width=800)
road_map.d_vect(map=roads, color="#272A2C", width=1)
road_map.d_vect(map=roads, where="highway IN ('secondary', 'secondary_link')", color="black", width=2)
road_map.d_vect(map=roads, where="highway IN ('primary', 'primary_link')", color="black", width=3)
road_map.d_vect(map=roads, where="highway IN ('trunk', 'motorway', 'trunk_link', 'motorway_link')", color="#b70003", width=3)
road_map.show()

In [ ]:
import pandas as pd
pd.DataFrame(tools.v_db_select(map=roads, format="json")["records"])

## Schools

Source: NC OneMap (2019). North Carolina Department of Information Technology, Government Data Analytics Center, Center for Geographic
Information and Analysis. Available at www.nconemap.gov.

In [ ]:
tools.g_extension(extension="v.in.ags")

In [ ]:
schools = "schools"

In [ ]:
tools.v_in_ags(url="https://services.nconemap.gov/secure/rest/services/NC1Map_Education/MapServer/3", output=schools, extent="region")

In [ ]:
pd.DataFrame(tools.v_db_select(map=schools, format="json")["records"])

In [ ]:
tools.v_support(map=schools, map_name="Public schools", organization="NC OneMap", person="NC OneMap")
tools.v_info(map=schools)

In [ ]:
school_map = gj.Map(width=800)
school_map.d_vect(map=roads, color="gray")
school_map.d_vect(map=schools, where="elem = 'yes'", size=15, icon="basic/pin", fill_color="green", legend_label="Elementary")
school_map.d_vect(map=schools, where="middle = 'yes'", size=15, icon="basic/pin", fill_color="yellow", legend_label="Middle")
school_map.d_vect(map=schools, where="high = 'yes'", size=15, icon="basic/pin", fill_color="orange", legend_label="High")
school_map.d_legend_vect(title="School type", flags="b", at=[2,40])
school_map.show()

## Hospitals

Source: NC OneMap (2019). North Carolina Department of Information Technology, Government Data Analytics Center, Center for Geographic
Information and Analysis. Available at www.nconemap.gov.

In [ ]:
hospitals = "hospitals"

In [ ]:
tools.v_in_ags(url="https://services.nconemap.gov/secure/rest/services/NC1Map_Health/MapServer/0", output=hospitals, extent="region")

In [ ]:
pd.DataFrame(tools.v_db_select(map=hospitals, format="json")["records"])

In [ ]:
tools.v_support(map=hospitals, map_name="Hospitals", organization="NC OneMap", person="NC OneMap")
tools.v_info(map=hospitals)

In [ ]:
school_map = gj.Map(width=800)
school_map.d_vect(map=roads, color="gray")
school_map.d_vect(map=hospitals, size=15, icon="basic/pin", fill_color="green")
school_map.show()

## Census blocks

In [ ]:
census_blocks = "census_blocks"

In [ ]:
!pip install pygris

In [ ]:
import pygris

blocks = pygris.blocks(state="37", county="183", year=2025, subset_by=bbox_latlon())
blocks.to_file("census_blocks.gpkg", driver="GPKG")
tools.v_import(input="census_blocks.gpkg", output=census_blocks)

In [ ]:
pd.DataFrame(tools.v_db_select(map=census_blocks, format="json")["records"])

In [ ]:
block_map = gj.Map(width=800)
block_map.d_rast(map=landuse)
block_map.d_vect(map=census_blocks, fill_color="none")
block_map.show()

In [ ]:
tools.v_support(map=census_blocks, map_name="Census blocks", organization="US Census Bureau", person="US Census Bureau")
tools.v_info(map=census_blocks)

## Municipal boundary

In [ ]:
municipal_boundary = "municipal_boundary"

In [ ]:
raleigh = pygris.places(state="37", year=2025).query("NAME == 'Raleigh'")
raleigh.to_file("raleigh.gpkg", driver="GPKG")
tools.v_import(input="raleigh.gpkg", output=municipal_boundary)

In [ ]:
block_map = gj.Map(width=800)
block_map.d_vect(map=municipal_boundary)
block_map.show()

In [ ]:
pd.DataFrame(tools.v_db_select(map=municipal_boundary, format="json")["records"])

In [ ]:
tools.v_support(map=municipal_boundary, map_name="Raleigh municipal boundary", organization="US Census Bureau", person="US Census Bureau")
tools.v_info(map=municipal_boundary)

## Zipcodes

In [ ]:
zipcodes = "zipcodes"

In [ ]:
zctas = pygris.zctas(state="37", year=2010, subset_by=bbox_latlon())
zctas.to_file("zipcodes.gpkg", driver="GPKG")
tools.v_import(input="zipcodes.gpkg", output=zipcodes)

In [ ]:
pd.DataFrame(tools.v_db_select(map=zipcodes, format="json")["records"])

In [ ]:
block_map = gj.Map(width=800)
block_map.d_rast(map=landuse)
block_map.d_vect(map=zipcodes, fill_color="none", color="black", width=2)
block_map.show()

In [ ]:
tools.v_support(map=zipcodes, map_name="ZIP Code Tabulation Areas", organization="US Census Bureau", person="US Census Bureau")
tools.v_info(map=zipcodes)

## Geology
macrostrat.org

In [ ]:
nc = pygris.states(year=2020).query("STATEFP == '37'")
nc.to_file("nc.gpkg", driver="GPKG")
tools.v_import(input="nc.gpkg", output="nc")

In [ ]:
import geopandas as gpd
import requests
from shapely.geometry import box

nc = pygris.states(year=2020).query("STATEFP == '37'")

response = requests.get(
    "https://macrostrat.org/api/v2/carto/small",
    params={"shape": box(*nc.total_bounds).wkt, "format": "geojson"},
)
response.raise_for_status()
features = response.json()["success"]["data"]["features"]

gdf = gpd.GeoDataFrame.from_features(features, crs="EPSG:4326")
gdf.to_file("geology.gpkg", driver="GPKG")

In [ ]:
geology = "geology"
tools.v_import(input="geology.gpkg", output=f"{geology}_")
tools.v_clip(input=f"{geology}_", output=geology, clip="nc")
tools.g_remove(name="nc,geology_", flags="f", type="vector")
tools.v_colors(map=geology, rgb_column="color", flags="c")

In [ ]:
block_map = gj.Map(width=800)
block_map.d_vect(map=geology)
block_map.show()

In [ ]:
pd.DataFrame(tools.v_db_select(map=geology, format="json")["records"])

In [ ]:
tools.v_support(map=geology, map_name="Geology", organization="Macrostrat", person="Macrostrat")
tools.v_info(map=geology)

## Soils
SSURGO database:
https://www.nrcs.usda.gov/resources/data-and-reports/soil-survey-geographic-database-ssurgo

In [ ]:
tools.g_extension(extension="r.in.ssurgo")

In [ ]:
soils = "soils"

In [ ]:
tools.r_in_ssurgo(soils="soils_")

In [ ]:
tools.v_clip(input="soils_", output=soils, flags="r")
tools.g_remove(name="region,soils_", flags="f", type="vector")

In [ ]:
m = gj.Map(width=800)
m.d_vect(map=soils, flags="c", color="none")
m.show()

In [ ]:
pd.DataFrame(tools.v_db_select(map=soils, format="json")["records"])

In [ ]:
tools.v_support(map=soils, map_name="SSURGO soils", organization="USDA", person="USDA")
tools.v_info(map=soils)

## Watersheds

USGS WBD - Watershed Boundary Dataset
https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer

In [ ]:
watersheds = "watersheds"

In [ ]:
tools.v_in_ags(url="https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6/", extent="region", output=watersheds, snap=1e-8)

In [ ]:
m = gj.InteractiveMap()
m.add_vector(watersheds)
m.show()

In [ ]:
pd.DataFrame(tools.v_db_select(map=watersheds, format="json")["records"])

In [ ]:
tools.v_support(map=watersheds, map_name="HUC12 subwatersheds", organization="USGS", person="USGS")
tools.v_info(map=watersheds)

## Lakes
Wake County GIS

In [ ]:
lakes = "lakes"

In [ ]:
tools.v_in_ags(url="https://services1.arcgis.com/a7CWfuGP5ZnLYE7I/arcgis/rest/services/WaterBodies/FeatureServer/0", extent="region", output="lakes_all", snap=1e-8)

In [ ]:
# There is an extra pond outside of the region, not sure how did it end up there
tools.v_extract(input="lakes_all", where="HYDRO_TYPE IN ('LAKE/POND', 'RESERVOIR') AND cat != 631", output=lakes)
tools.g_remove(name="lakes_all", flags="f", type="vector")

In [ ]:
pd.DataFrame(tools.v_db_select(map=lakes, format="json")["records"])

In [ ]:
m = gj.Map(width=800)
m.d_rast(map=elevation)
m.d_vect(map=lakes, fill_color="aqua", color="none")
m.show()

In [ ]:
tools.v_support(map=lakes, map_name="Lakes, ponds and reservoirs", organization="Wake County GIS", person="Wake County GIS")
tools.v_info(map=lakes)

## Streams
Wake County GIS

In [ ]:
streams = "streams"

In [ ]:
tools.v_in_ags(url="https://services1.arcgis.com/a7CWfuGP5ZnLYE7I/arcgis/rest/services/Hydrolines/FeatureServer/0/", extent="region", output=streams)

In [ ]:
m = gj.Map(width=800)
m.d_rast(map=elevation)
m.d_vect(map=lakes, fill_color="aqua", color="none")
m.d_vect(map=streams, color="aqua")
m.d_vect(map=watersheds, fill_color="none", color="brown", width=3)
m.show()

In [ ]:
pd.DataFrame(tools.v_db_select(map=streams, format="json")["records"])

In [ ]:
tools.v_support(map=streams, map_name="Streams and rivers", organization="Wake County GIS", person="Wake County GIS")
tools.v_info(map=streams)

## Add new mapset

In [ ]:
gs.create_mapset(name="mapset_1")